# PageServe on a Colab GPU

Clone, install, verify, then benchmark the engine on a **T4 / L4 / A100**.

**Before you start:** `Runtime ▸ Change runtime type ▸` pick a GPU. Everything below runs top to bottom with no edits.

| Step | What it does | Time on a T4 |
|---|---|---|
| 1–2 | Clone the repo, install dependencies | ~1 min |
| 3 | Unit tests (CPU, tiny models) | ~1 min |
| 4 | GPU smoke test: exactness on CUDA + real-model sanity check | ~2 min |
| 5 | Offline ablation: batching, prefix caching, speculative decoding | ~10–15 min |
| 6 | Serving benchmark: Poisson load sweep over HTTP, streaming | ~25 min (`QUICK=True`: ~8 min) |
| 7 | Charts + tables, download results | seconds |

The benchmark model is chosen from GPU memory: T4 → `Qwen2.5-1.5B-Instruct`, L4 → `Qwen2.5-3B-Instruct`, A100 → `Qwen2.5-7B-Instruct` (override with `MODEL`).

In [ ]:
# ── Settings ──────────────────────────────────────────────────────────────
REPO_URL = "https://github.com/TryingtobeingNikhil/vLLM_Inference_Engine.git"
BRANCH = "engine/gpu-upgrade"   # use "main" once this branch is merged
MODEL = "auto"                  # or e.g. "Qwen/Qwen2.5-3B-Instruct"
QUICK = False                   # True = smaller benchmark runs
DRAFT_MODEL = "auto"            # draft for speculative decoding ("" to skip; best on A100 with a 7B target)

!nvidia-smi

## 1. Clone

In [ ]:
import os, subprocess, sys
if not os.path.exists("vLLM_Inference_Engine"):
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, "vLLM_Inference_Engine"])
    if r.returncode != 0:
        # Don't silently fall back to another branch: it may not contain this notebook's code.
        raise SystemExit(f"Could not clone branch {BRANCH!r}. Push it to GitHub first, "
                         f"or set BRANCH = 'main' once it has been merged.")
%cd vLLM_Inference_Engine
!git log --oneline -1
assert os.path.exists("scripts/colab_setup.sh"), "This checkout predates the Colab tooling — check BRANCH."


## 2. Install

In [ ]:
!bash scripts/colab_setup.sh

## 3. Unit tests

Tiny randomly-initialised models check the engine token-for-token against HuggingFace `generate()` under every feature combination.

In [ ]:
!python -m pytest -q -m "not slow"

## 4. GPU smoke test

1. **Exactness on CUDA** — a tiny float64 model through the full scheduler on the GPU (paged writes/gathers, prefix caching, swap *and* recompute preemption, n-gram and draft speculation) must match HF `generate()` exactly.
2. **Real model** — engine vs HF greedy on the benchmark model (half precision: late divergence is normal).

In [ ]:
!python scripts/gpu_smoke_test.py --model {MODEL}

## 5. Offline ablation (in-process, no HTTP)

Same seeded workload through each system, all requests submitted at once:

* **batching** — HF sequential (Phase 1) vs HF static batching vs engine without batching vs continuous batching
* **prefix** — shared 1k-token system prompt, prefix caching off vs on
* **spec** — copy-heavy and random prompts, no speculation vs n-gram vs draft model

In [ ]:
quick = "--num-requests 48 --sequential-requests 6" if QUICK else ""
draft = f"--draft-model {DRAFT_MODEL}" if DRAFT_MODEL else ""
!python -m benchmarks.bench_offline --model {MODEL} {draft} {quick}

## 6. Serving benchmark (HTTP, streaming, Poisson arrivals)

Each configuration gets its own server process; the client measures TTFT / TPOT / ITL / E2E from the *scheduled* send time and samples GPU utilisation.

Configs: `sequential` (Phase 1), `continuous` (batching + paged KV), `prefix` (+ prefix caching), `ngram` (+ n-gram speculation).

In [ ]:
if QUICK:
    !python -m benchmarks.bench_serving --model {MODEL} --num-requests 80 --sequential-requests 12 --rates 4 16 inf
else:
    !python -m benchmarks.bench_serving --model {MODEL}

Prefix caching under load (optional): a long shared system prompt is where it pays off.

In [ ]:
!python -m benchmarks.bench_serving --model {MODEL} --configs continuous,prefix --workload shared_prefix --rates 4 8 16 --num-requests 120

## 7. Results

In [ ]:
import glob
from IPython.display import Image, Markdown, display

!python -m benchmarks.plot_results results/*.json
for md_path in sorted(glob.glob("results/*.md")):
    display(Markdown(open(md_path).read()))
for png in sorted(glob.glob("results/*.png")):
    display(Image(png))

In [ ]:
# Download everything (JSON + Markdown + charts + server logs)
!zip -qr pageserve_results.zip results
from google.colab import files
files.download("pageserve_results.zip")

## Appendix: talk to the server yourself

Start the engine in the background, then stream a completion. Any env var from `inference_engine/config.py` works (e.g. `SPECULATIVE_METHOD=ngram`, `MAX_BATCH_SIZE=128`).

In [ ]:
import subprocess, time, requests
from benchmarks.common import pick_model
server = subprocess.Popen(
    f"MODEL_NAME={pick_model(MODEL)} python -m uvicorn inference_engine.server.app_v2:app --port 8001 > server.log 2>&1",
    shell=True)
for _ in range(300):
    try:
        if requests.get("http://127.0.0.1:8001/health", timeout=2).ok: break
    except Exception: pass
    time.sleep(2)
print(requests.get("http://127.0.0.1:8001/health").json())

In [ ]:
import json, requests
with requests.post("http://127.0.0.1:8001/generate", stream=True,
                   json={"prompt": "Explain paged attention in two sentences.", "max_new_tokens": 96, "stream": True}) as r:
    for line in r.iter_lines():
        if line.startswith(b"data: ") and line != b"data: [DONE]":
            event = json.loads(line[6:])
            if "text" in event: print(event["text"], end="", flush=True)
            elif event.get("done"): print(f"\n\nTTFT {event['ttft_ms']:.0f} ms, TPOT {event['tpot_ms']:.1f} ms")

In [ ]:
# Load test against the running server with the same client used by the benchmarks
!python run_load_test.py --server phase2 --profile poisson --rps 8 --workload random --num-requests 100 --stream --ignore-eos
server.terminate()